In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/house-prices-advanced-regression-techniques/sample_submission.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/data_description.txt
/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv


In [8]:
import pandas as pd

df = pd.read_csv(
    "/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv"
)

In [9]:
df.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [10]:
df.shape

(1460, 81)

# Ridge Regression — House Price Prediction

In this notebook, we will implement and understand **Ridge Regression**
using the House Prices dataset from Kaggle.

### Objectives

- Understand why regularization is needed in Linear Regression
- Understand the concept of Ridge Regression
- Apply Ridge Regression to a real-world dataset
- Compare Linear Regression and Ridge Regression
- Understand the effect of the regularization parameter `alpha`
- Analyze how Ridge Regression affects model coefficients
- Implement Ridge Regression using Scikit-learn
- Later, implement Ridge Regression using Gradient Descent

## Why Ridge Regression?

Linear Regression tries to minimize the sum of squared errors between the
actual and predicted values.

However, when a dataset contains:

- Many features
- Highly correlated features
- Features with different scales
- Multicollinearity

the Linear Regression model can produce large coefficients and may
overfit the training data.

Ridge Regression addresses this problem by adding a **regularization
penalty** to the Linear Regression cost function.

The penalty discourages the model from assigning excessively large
coefficients to the features.

The regularization parameter is called **alpha (α)**.

A larger value of α applies stronger regularization, while a smaller
value applies weaker regularization.

In [16]:
print("Number of rows:",df.shape[0])
print("number of columns: ",df.shape[1])

Number of rows: 1460
number of columns:  81


In [17]:
df.dtypes

Id                 int64
MSSubClass         int64
MSZoning          object
LotFrontage      float64
LotArea            int64
                  ...   
MoSold             int64
YrSold             int64
SaleType          object
SaleCondition     object
SalePrice          int64
Length: 81, dtype: object

In [18]:
df.isnull().sum().sort_values(ascending=False).head(20)

PoolQC          1453
MiscFeature     1406
Alley           1369
Fence           1179
MasVnrType       872
FireplaceQu      690
LotFrontage      259
GarageQual        81
GarageFinish      81
GarageType        81
GarageYrBlt       81
GarageCond        81
BsmtFinType2      38
BsmtExposure      38
BsmtCond          37
BsmtQual          37
BsmtFinType1      37
MasVnrArea         8
Electrical         1
Condition2         0
dtype: int64

## 3. Feature Selection

The original dataset contains many numerical and categorical features.

For our first Ridge Regression implementation, we will focus on a selected
set of numerical features.

This allows us to concentrate on understanding:

- Multiple Linear Regression
- Feature scaling
- Ridge regularization
- The effect of the regularization parameter α
- Changes in model coefficients

without introducing the additional complexity of categorical encoding.

### Selected Features

We will use the following numerical features:

- `GrLivArea` — Above-ground living area
- `YearBuilt` — Original construction year
- `OverallQual` — Overall material and finish quality
- `OverallCond` — Overall condition of the house
- `GarageArea` — Garage area
- `TotalBsmtSF` — Total basement area
- `1stFlrSF` — First-floor area
- `2ndFlrSF` — Second-floor area
- `FullBath` — Number of full bathrooms
- `BedroomAbvGr` — Number of bedrooms above ground
- `TotRmsAbvGrd` — Total rooms above ground

The target variable is `SalePrice`.

In [20]:
features = [
    'GrLivArea',
    'YearBuilt',
    'OverallQual',
    'OverallCond',
    'GarageArea',
    'TotalBsmtSF',
    '1stFlrSF',
    '2ndFlrSF',
    'FullBath',
    'BedroomAbvGr',
    'TotRmsAbvGrd'
]

X = df[features]
y = df['SalePrice']

In [21]:
X.head()
y.shape

(1460,)

In [22]:
X.isnull().sum()

GrLivArea       0
YearBuilt       0
OverallQual     0
OverallCond     0
GarageArea      0
TotalBsmtSF     0
1stFlrSF        0
2ndFlrSF        0
FullBath        0
BedroomAbvGr    0
TotRmsAbvGrd    0
dtype: int64

## 4. Train-Test Split

We divide the dataset into two parts:

- **Training set** — used to train the model
- **Testing set** — used to evaluate the model on unseen data

We will use an **80:20 split**:

- 80% of the data → Training
- 20% of the data → Testing

The test data must remain unseen during model training.

We use `random_state=42` so that the same split can be reproduced
whenever we run the notebook.

In [23]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(
    X,y,test_size= 0.2,random_state=42
)

In [24]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (1168, 11)
X_test: (292, 11)
y_train: (1168,)
y_test: (292,)


## 5. Linear Regression Baseline

Before applying Ridge Regression, we first build a standard Linear Regression
model.

This gives us a baseline against which we can compare Ridge Regression.

Linear Regression tries to find coefficients that minimize the **sum of
squared errors** between the actual and predicted target values.

The model can be represented as:

\[
\hat{y} = \beta_0 + \beta_1x_1 + \beta_2x_2 + \cdots + \beta_nx_n
\]

where:

- \(\hat{y}\) = predicted value
- \(\beta_0\) = intercept
- \(\beta_1, \beta_2, ..., \beta_n\) = model coefficients
- \(x_1, x_2, ..., x_n\) = input features

We will use this model as our baseline before introducing Ridge
regularization.

In [25]:
from sklearn.linear_model import LinearRegression

In [26]:
lr_model = LinearRegression()
lr_model.fit(X_train,y_train)

LinearRegression()

In [28]:
y_pred_lr = lr_model.predict(X_test)

In [29]:
from sklearn.metrics import mean_squared_error,r2_score

mse_lr = mean_squared_error(y_test, y_pred_lr)
rmse_lr = mean_squared_error(y_test, y_pred_lr) ** 0.5
r2_lr = r2_score(y_test, y_pred_lr)

print("Linear Regression")
print("------------------")
print("MSE :", mse_lr)
print("RMSE:", rmse_lr)
print("R²  :", r2_lr)

Linear Regression
------------------
MSE : 1440347163.077043
RMSE: 37951.905921535
R²  : 0.8122182650162346


### Evaluation Metrics

We use three metrics to evaluate the regression model:

**Mean Squared Error (MSE)**

MSE calculates the average squared difference between actual and predicted
values. Lower MSE indicates smaller prediction errors.

**Root Mean Squared Error (RMSE)**

RMSE is the square root of MSE. It is expressed in the same unit as the
target variable, which makes it easier to interpret.

Lower RMSE indicates better predictive performance.

**R² Score**

R² measures how much of the variation in the target variable is explained
by the model.

A value closer to 1 indicates that the model explains more of the variation
in the target.

In [31]:
coefficients = pd.DataFrame({
    'Feature': X_train.columns,
    'Coefficient': lr_model.coef_
})

coefficients.sort_values(
    by='Coefficient',
    key=abs,
    ascending=False
)

,Feature,Coefficient
2,OverallQual,18644.511641
9,BedroomAbvGr,-9315.282884
3,OverallCond,6518.825700
10,TotRmsAbvGrd,4218.016566
8,FullBath,-2844.861550
1,YearBuilt,502.762093
4,GarageArea,40.575938
0,GrLivArea,32.469572
6,1stFlrSF,29.023769
7,2ndFlrSF,17.608417


## 6. Feature Scaling

Our selected features have very different ranges.

For example:

- `YearBuilt` is measured in years
- `GrLivArea` is measured in square feet
- `OverallQual` has relatively small integer values
- `GarageArea` is measured in square feet

Ridge Regression adds a penalty based on the magnitude of the model
coefficients.

Therefore, feature scaling is important because we want the features to
be on a comparable scale before applying regularization.

We will use **Standardization**.

Standardization transforms each feature using:

\[
z = \frac{x-\mu}{\sigma}
\]

where:

- \(x\) = original value
- \(\mu\) = mean of the feature
- \(\sigma\) = standard deviation of the feature
- \(z\) = standardized value

After standardization, each feature will approximately have:

- Mean = 0
- Standard deviation = 1

### Important

The scaler is fitted **only on the training data**.

The same transformation is then applied to the test data.

This prevents information from the test set from leaking into the training
process.

In [33]:
from sklearn.preprocessing import StandardScaler

In [34]:
scaler = StandardScaler()

In [39]:
X_train_scaled =  scaler.fit_transform(X_train)

In [37]:
X_test_scaled = scaler.transform(X_test)

In [40]:
X_train_scaled[:5]

array([[-0.40709315, -0.45546896, -0.82044456,  0.3722173 , -0.86383727,
         0.57261219,  0.37423523, -0.80192292, -1.05556573,  0.13621832,
        -0.96456591],
       [ 0.08317013,  0.71860895, -0.08893368,  1.26860866, -0.45626397,
        -0.59654659, -0.95820221,  0.9550877 ,  0.7736639 ,  0.13621832,
         0.27075534],
       [-1.39525026, -1.98829291, -0.82044456,  1.26860866, -2.25716927,
        -0.60335723, -0.96596398, -0.80192292, -1.05556573, -1.10677385,
        -1.58222654],
       [ 0.45897505, -1.10773447, -0.82044456,  1.26860866, -1.11975539,
        -0.75092096, -0.48732141,  0.9892265 , -1.05556573,  0.13621832,
         0.27075534],
       [ 0.31208684, -1.53170705, -0.82044456,  0.3722173 , -0.79748813,
        -0.08120864, -0.37089483,  0.7115642 ,  0.7736639 ,  0.13621832,
        -0.34690528]])

In [41]:
X_test_scaled[:5]

array([[-0.87637239, -0.25978931, -0.08893368,  2.16500001, -1.00601401,
        -0.00629167, -0.26223003, -0.80192292, -1.05556573,  0.13621832,
        -0.34690528],
       [ 2.08809894,  0.75122223,  1.37408806, -0.52417405,  1.11715856,
         0.91087367,  0.85546506,  1.75165987,  0.7736639 ,  0.13621832,
         1.50607659],
       [-0.95267796, -1.43386723, -0.82044456,  0.3722173 , -0.55104846,
        -0.12207245, -0.36572032, -0.80192292, -1.05556573, -1.10677385,
        -0.96456591],
       [ 0.26058058, -0.78160172, -0.08893368,  1.26860866, -0.26669499,
        -0.13115329, -0.42781449,  0.7001846 ,  0.7736639 ,  0.13621832,
         0.27075534],
       [ 0.17664445,  1.17519481,  2.10559893, -0.52417405,  2.06500345,
         1.26729683,  1.16593592, -0.80192292,  0.7736639 , -1.10677385,
        -0.34690528]])

In [ ]:
## 7. Ridge Regression

Ridge Regression is a regularized version of Linear Regression.

In ordinary Linear Regression, the model minimizes the sum of squared
prediction errors.

Ridge Regression adds an additional penalty to the cost function based on
the squared magnitude of the model coefficients.

The Ridge objective function is:

\[
J(\beta) =
\sum_{i=1}^{n}(y_i-\hat{y}_i)^2
+
\alpha\sum_{j=1}^{p}\beta_j^2
\]

where:

- \(y_i\) = actual value
- \(\hat{y}_i\) = predicted value
- \(\beta_j\) = model coefficients
- \(\alpha\) = regularization strength

The second term is called the **L2 regularization penalty**.

### Role of α

The parameter \(\alpha\) controls how strongly the coefficients are
penalized.

- **α = 0** → Ridge becomes ordinary Linear Regression
- **Small α** → weak regularization
- **Large α** → strong regularization

Ridge Regression generally reduces the magnitude of coefficients rather
than forcing them exactly to zero.

This can help reduce the effect of multicollinearity and control
overfitting.

In [42]:
from sklearn.linear_model import Ridge

In [43]:
ridge_model = Ridge(alpha = 1)

In [44]:
ridge_model.fit(X_train_scaled,y_train)

Ridge(alpha=1)

In [46]:
y_pred_ridge = ridge_model.predict(X_test_scaled)

In [47]:
mse_ridge = mean_squared_error(y_test, y_pred_ridge)
rmse_ridge = mean_squared_error(y_test, y_pred_ridge) ** 0.5
r2_ridge = r2_score(y_test, y_pred_ridge)

print("Ridge Regression")
print("----------------")
print("MSE :", mse_ridge)
print("RMSE:", rmse_ridge)
print("R²  :", r2_ridge)

Ridge Regression
----------------
MSE : 1440488573.1764455
RMSE: 37953.7688929103
R²  : 0.8121998290207394


In [48]:
ridge_coefficients = pd.DataFrame({
    'Feature': X_train.columns,
    'Coefficient': ridge_model.coef_
})

ridge_coefficients.sort_values(
    by='Coefficient',
    key=abs,
    ascending=False
)

,Feature,Coefficient
2,OverallQual,25467.906260
0,GrLivArea,16795.682114
1,YearBuilt,15388.974714
6,1stFlrSF,11346.790396
4,GarageArea,8572.490235
7,2ndFlrSF,7901.339480
9,BedroomAbvGr,-7484.018296
3,OverallCond,7261.023129
5,TotalBsmtSF,7139.707659
10,TotRmsAbvGrd,6832.731718


## 8. Effect of α (Regularization Strength)

The parameter **α (alpha)** controls the strength of L2 regularization
in Ridge Regression.

A small value of α applies weak regularization, while a large value
applies stronger regularization.

We will train multiple Ridge Regression models using different values
of α and observe:

- RMSE
- R² Score
- Magnitude of the coefficients

We will experiment with:

- α = 0.01
- α = 0.1
- α = 1
- α = 10
- α = 100

### Expected behavior

As α increases, the regularization penalty becomes stronger.

Therefore, the model coefficients generally become smaller in magnitude.

However, increasing α does not automatically improve model performance.
An excessively large α can cause the model to become too constrained
and may lead to underfitting.

In [49]:
alphas = [0.01, 0.1, 1, 10, 100]

results = []

for alpha in alphas:

    model = Ridge(alpha=alpha)

    model.fit(X_train_scaled, y_train)

    y_pred = model.predict(X_test_scaled)

    mse = mean_squared_error(y_test, y_pred)
    rmse = mean_squared_error(y_test, y_pred) ** 0.5
    r2 = r2_score(y_test, y_pred)

    results.append({
        'Alpha': alpha,
        'MSE': mse,
        'RMSE': rmse,
        'R2': r2
    })

In [50]:
results_df = pd.DataFrame(results)
results_df

,Alpha,MSE,RMSE,R2
0,0.01,1.440348e+09,37951.920404,0.812218
1,0.10,1.440359e+09,37952.055305,0.812217
2,1.00,1.440489e+09,37953.768893,0.812200
3,10.00,1.442676e+09,37982.571790,0.811915
4,100.00,1.476027e+09,38419.102381,0.807567


In [52]:
coefficient_results = {}

for alpha in alphas:

    model = Ridge(alpha=alpha)

    model.fit(X_train_scaled, y_train)

    coefficient_results[alpha] = model.coef_

In [53]:
coefficients_df = pd.DataFrame(
    coefficient_results,
    index=X_train.columns
)

coefficients_df

,0.01,0.10,1.00,10.00,100.00
GrLivArea,17018.068792,16993.908335,16795.682114,16077.347214,14476.159207
YearBuilt,15415.592403,15413.084378,15388.974714,15181.173371,13719.597332
OverallQual,25487.480238,25485.721371,25467.906260,25285.005061,23641.610340
OverallCond,7272.183053,7271.145298,7261.023129,7168.164243,6376.849618
GarageArea,8561.833415,8562.805364,8572.490235,8665.642005,9313.622778
TotalBsmtSF,7117.175821,7119.210160,7139.707659,7339.841403,8530.936494
1stFlrSF,11219.622160,11234.102383,11346.790396,11550.194381,10545.852887
2ndFlrSF,7738.893076,7756.973593,7901.339480,8288.407325,7957.442088
FullBath,-1554.939689,-1552.381193,-1526.734003,-1274.314522,652.540297
BedroomAbvGr,-7494.139275,-7493.223229,-7484.018296,-7390.647240,-6498.191028


### Interpretation

As α increases, Ridge Regression applies a stronger L2 penalty.

This generally causes the magnitude of the coefficients to decrease.

For example, if a coefficient changes from:

β = 50,000

to:

β = 35,000

after increasing α, the model has reduced the influence of that feature.

However, Ridge Regression usually does **not** make coefficients exactly
zero.

This is an important difference between Ridge and Lasso Regression.

- **Ridge → shrinks coefficients toward zero**
- **Lasso → can shrink coefficients exactly to zero**

Therefore, Ridge is useful when we want to control coefficient magnitude
while generally retaining all features in the model.

## 10. Linear Regression vs Ridge Regression

We now compare the baseline Linear Regression model with Ridge Regression.

The main purpose is to observe whether adding L2 regularization changes
the model's prediction performance.

In [54]:
comparison = pd.DataFrame({
    'Model': ['Linear Regression', 'Ridge Regression'],
    'RMSE': [rmse_lr, rmse_ridge],
    'R2': [r2_lr, r2_ridge]
})

comparison

,Model,RMSE,R2
0,Linear Regression,37951.905922,0.812218
1,Ridge Regression,37953.768893,0.812200
